In [4]:
import os
import time
import requests
import pandas as pd
import numpy as np
import psycopg2
import psycopg2.extras
from datetime import datetime, timedelta
from dotenv import load_dotenv

# ── Configuración ──────────────────────────────────────────────────────────────
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
POLYGON_API_KEY   = os.getenv("POLYGON_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, POLYGON_API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

RUN_ID     = datetime.now().strftime("%Y%m%d_%H%M")
BENCHMARK  = "SPY"
DIAS_HIST  = 730    # historia inicial: 2 años
DIAS_MIN   = 252    # mínimo para calcular indicadores (1 año)

# ── Rate limiting ──────────────────────────────────────────────────────────────
# Options Starter: ~10-15 req/min según reportes de usuarios
# Usamos 5 segundos entre requests (12 req/min) para estar seguros
SLEEP_ENTRE_REQUESTS = 5       # segundos entre cada request
MAX_REINTENTOS       = 3       # reintentos si hay 429
SLEEP_EN_429         = 60      # esperar 60s si Polygon rechaza


# ── Conexión ───────────────────────────────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD, host=POSTGRES_HOST, port=POSTGRES_PORT,
    )


# ── Leer tickers activos del catálogo ─────────────────────────────────────────
def leer_tickers(conn) -> list[dict]:
    sql = """
        SELECT ticker, nombre, tipo, sector_gics, sector_etf, industria
        FROM sector.sector_etfs
        WHERE activo = TRUE
        ORDER BY tipo, sector_gics, ticker
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql)
        return [dict(r) for r in cur.fetchall()]


# ── Determinar desde qué fecha descargar ──────────────────────────────────────
def ultima_fecha_en_db(conn, ticker: str):
    """
    Devuelve la última fecha que ya tenemos en sector_raw para este ticker.
    Si no hay datos, devuelve None → hay que bajar el histórico completo.
    """
    sql = "SELECT MAX(fecha) FROM sector.sector_raw WHERE ticker = %s"
    with conn.cursor() as cur:
        cur.execute(sql, (ticker,))
        resultado = cur.fetchone()[0]
        return resultado


# ── Descarga desde Polygon ────────────────────────────────────────────────────
def descargar_precios(ticker: str, desde: str, hasta: str) -> pd.DataFrame | None:
    url = (
        f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day"
        f"/{desde}/{hasta}"
        f"?adjusted=true&sort=asc&limit=5000&apiKey={POLYGON_API_KEY}"
    )

    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            r = requests.get(url, timeout=15)

            # 429 → esperar y reintentar
            if r.status_code == 429:
                print(f"\n  ⏳ Rate limit en {ticker} (intento {intento}/{MAX_REINTENTOS}) "
                      f"— esperando {SLEEP_EN_429}s...")
                time.sleep(SLEEP_EN_429)
                continue

            r.raise_for_status()
            data = r.json()

            if "results" not in data or not data["results"]:
                print(f"  ⚠ Sin datos: {ticker}")
                return None

            df = pd.DataFrame(data["results"])
            df["fecha"] = pd.to_datetime(df["t"], unit="ms").dt.date
            df = df.rename(columns={"o":"open","h":"high","l":"low",
                                     "c":"close","v":"volume","vw":"vwap"})
            cols = ["fecha","open","high","low","close","volume","vwap"]
            cols_presentes = [c for c in cols if c in df.columns]

            # Sleep entre requests para respetar rate limit
            time.sleep(SLEEP_ENTRE_REQUESTS)
            return df[cols_presentes].copy()

        except Exception as e:
            if intento < MAX_REINTENTOS:
                print(f"\n  ⚠ Error {ticker} (intento {intento}): {e} — reintentando...")
                time.sleep(SLEEP_EN_429)
            else:
                print(f"  ✗ Error descargando {ticker}: {e}")
                return None

    return None


# ── INSERT en sector_raw ───────────────────────────────────────────────────────
def insertar_precios(conn, ticker: str, df: pd.DataFrame):
    if df is None or df.empty:
        return 0

    sql = """
        INSERT INTO sector.sector_raw
            (ticker, fecha, open, high, low, close, volume, vwap, run_id)
        VALUES
            (%(ticker)s, %(fecha)s, %(open)s, %(high)s, %(low)s,
             %(close)s, %(volume)s, %(vwap)s, %(run_id)s)
        ON CONFLICT (ticker, fecha) DO NOTHING
    """
    filas = []
    for _, row in df.iterrows():
        filas.append({
            "ticker":  ticker,
            "fecha":   row["fecha"],
            "open":    float(row["open"])   if "open"   in row else None,
            "high":    float(row["high"])   if "high"   in row else None,
            "low":     float(row["low"])    if "low"    in row else None,
            "close":   float(row["close"]),
            "volume":  int(row["volume"])   if "volume" in row else None,
            "vwap":    float(row["vwap"])   if "vwap"   in row else None,
            "run_id":  RUN_ID,
        })

    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, sql, filas)
    conn.commit()
    return len(filas)


# ── Cargar histórico completo desde DB ────────────────────────────────────────
def cargar_serie(conn, ticker: str) -> pd.DataFrame:
    """
    Lee todos los cierres históricos de sector_raw para un ticker.
    Se usa para calcular los indicadores sobre datos ya persistidos.
    """
    sql = """
        SELECT fecha, close, volume
        FROM sector.sector_raw
        WHERE ticker = %s
        ORDER BY fecha ASC
    """
    with conn.cursor() as cur:
        cur.execute(sql, (ticker,))
        rows = cur.fetchall()

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows, columns=["fecha","close","volume"])
    df["fecha"]  = pd.to_datetime(df["fecha"])
    df["close"]  = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df = df.set_index("fecha")
    return df


# ── Indicadores ────────────────────────────────────────────────────────────────
def calcular_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window).mean()
    loss  = -delta.clip(upper=0).rolling(window).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def calcular_slope(series: pd.Series, window: int = 5) -> float:
    s = series.dropna()
    if len(s) < window:
        return np.nan
    y = s.values[-window:]
    x = np.arange(window)
    return float(np.polyfit(x, y, 1)[0])


def calcular_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direccion = np.sign(close.diff())
    obv = (direccion * volume).fillna(0).cumsum()
    return obv


def calcular_retorno(series: pd.Series, dias: int) -> float:
    s = series.dropna()
    if len(s) < dias + 1:
        return np.nan
    return float((s.iloc[-1] / s.iloc[-dias - 1]) - 1) * 100


def calcular_percentil_52w(series: pd.Series) -> float:
    s = series.dropna()
    if len(s) < 252:
        return np.nan
    ventana = s.iloc[-252:]
    valor_actual = ventana.iloc[-1]
    return float(pd.Series(ventana).rank(pct=True).iloc[-1] * 100)


def calcular_dist_maximo_52w(series: pd.Series) -> float:
    s = series.dropna()
    if len(s) < 252:
        return np.nan
    max_52w = s.iloc[-252:].max()
    return float((s.iloc[-1] / max_52w - 1) * 100)


def calcular_indicadores(df_ticker: pd.DataFrame, df_spy: pd.DataFrame) -> dict | None:
    """
    Recibe el DataFrame del ticker y del benchmark (SPY).
    Devuelve un dict con todos los indicadores calculados.
    """
    if df_ticker.empty or df_spy.empty:
        return None

    # Alinear fechas con SPY
    df = df_ticker.join(df_spy[["close"]].rename(columns={"close":"close_spy"}), how="inner")

    if len(df) < DIAS_MIN:
        return None

    close     = df["close"]
    volume    = df["volume"]
    close_spy = df["close_spy"]

    # ── Fuerza relativa vs SPY
    rs = close / close_spy

    # RS semanal (resample a viernes)
    rs_weekly = rs.resample("W-FRI").last()

    rsi_rs_diario   = calcular_rsi(rs).iloc[-1]
    rsi_rs_semanal  = calcular_rsi(rs_weekly).iloc[-1]
    slope_rs        = calcular_slope(rs_weekly)
    rs_percentil    = calcular_percentil_52w(rs)

    # ── Retornos de precio absoluto
    ret_1m  = calcular_retorno(close, 21)
    ret_3m  = calcular_retorno(close, 63)
    ret_6m  = calcular_retorno(close, 126)
    ret_1a  = calcular_retorno(close, 252)

    # ── Distancia al máximo 52W
    dist_max_52w = calcular_dist_maximo_52w(close)

    # ── Volumen
    vol_ratio = float(volume.iloc[-1] / volume.rolling(20).mean().iloc[-1]) if len(volume) >= 20 else np.nan

    # ── OBV y su slope
    obv       = calcular_obv(close, volume)
    obv_slope = calcular_slope(obv)

    # ── RSI del precio (daily)
    rsi_precio = calcular_rsi(close).iloc[-1]

    return {
        "rs_vs_spy":      float(rs.iloc[-1]),
        "rsi_rs_diario":  float(rsi_rs_diario)  if not np.isnan(rsi_rs_diario)  else None,
        "rsi_rs_semanal": float(rsi_rs_semanal) if not np.isnan(rsi_rs_semanal) else None,
        "slope_rs":       float(slope_rs)        if not np.isnan(slope_rs)       else None,
        "rs_percentil":   float(rs_percentil)    if not np.isnan(rs_percentil)   else None,
        "ret_1m":         float(ret_1m)           if not np.isnan(ret_1m)         else None,
        "ret_3m":         float(ret_3m)           if not np.isnan(ret_3m)         else None,
        "ret_6m":         float(ret_6m)           if not np.isnan(ret_6m)         else None,
        "ret_1a":         float(ret_1a)           if not np.isnan(ret_1a)         else None,
        "dist_max_52w":   float(dist_max_52w)     if not np.isnan(dist_max_52w)   else None,
        "vol_ratio":      float(vol_ratio)         if not np.isnan(vol_ratio)      else None,
        "obv_slope":      float(obv_slope)         if not np.isnan(obv_slope)      else None,
        "rsi_precio":     float(rsi_precio)        if not np.isnan(rsi_precio)     else None,
        "close":          float(close.iloc[-1]),
        "fecha_ultimo":   df.index[-1].date(),
    }


# ── INSERT en sector_snapshot ──────────────────────────────────────────────────
def guardar_snapshot(conn, resultados: list[dict]):
    """
    Inserta una fila por ticker en sector_snapshot.
    Lee el estado_macro actual de macro.macro_diagnostico.
    """
    # Leer estado macro vigente
    sql_macro = """
        SELECT estado_macro
        FROM macro.macro_diagnostico
        ORDER BY calculado_en DESC
        LIMIT 1
    """
    with conn.cursor() as cur:
        cur.execute(sql_macro)
        row = cur.fetchone()
        estado_macro = row[0] if row else None

    # Tabla de alineación régimen → sectores favorecidos
    ALINEACION = {
        "EXPANSION":   ["XLK","XLY","XLF","XLI","SOXX","IGV","SKYY","XRT","IAI","ITA"],
        "SLOWDOWN":    ["XLV","XLP","XLU","GLD","TLT","IBB","PBJ","PHO","UTG","PUI"],
        "CONTRACTION": ["XLP","XLU","XLV","GLD","TLT","PBJ","PHO","UTG","IBB"],
        "RECOVERY":    ["XLI","XLB","XLF","XLK","PAVE","COPX","KBE","KRE","ITA"],
    }
    favorecidos = ALINEACION.get(estado_macro, [])

    sql_insert = """
        INSERT INTO sector.sector_snapshot (
            run_id, fecha_ultimo, ticker, tipo, sector_gics, sector_etf, industria,
            close, rs_vs_spy, rsi_rs_diario, rsi_rs_semanal, slope_rs, rs_percentil,
            ret_1m, ret_3m, ret_6m, ret_1a, dist_max_52w,
            vol_ratio, obv_slope, rsi_precio,
            estado_macro, alineacion_macro
        ) VALUES (
            %(run_id)s, %(fecha_ultimo)s, %(ticker)s, %(tipo)s,
            %(sector_gics)s, %(sector_etf)s, %(industria)s,
            %(close)s, %(rs_vs_spy)s, %(rsi_rs_diario)s, %(rsi_rs_semanal)s,
            %(slope_rs)s, %(rs_percentil)s,
            %(ret_1m)s, %(ret_3m)s, %(ret_6m)s, %(ret_1a)s, %(dist_max_52w)s,
            %(vol_ratio)s, %(obv_slope)s, %(rsi_precio)s,
            %(estado_macro)s, %(alineacion_macro)s
        )
    """

    filas = []
    for r in resultados:
        alineacion = "ALIGNED" if r["ticker"] in favorecidos else "NEUTRAL"
        filas.append({
            **r,
            "estado_macro":     estado_macro,
            "alineacion_macro": alineacion,
            "fecha_ultimo":     r.get("fecha_ultimo"),
        })

    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, sql_insert, filas)
    conn.commit()
    print(f"  ✓ {len(filas)} filas guardadas en sector_snapshot (run_id: {RUN_ID})")
    print(f"  ✓ Estado macro vigente: {estado_macro}")
    aligned = sum(1 for f in filas if f["alineacion_macro"] == "ALIGNED")
    print(f"  ✓ ETFs alineados con régimen: {aligned}/{len(filas)}")


# ── Pipeline principal ─────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print(f"  SECTOR PRECIOS — {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  run: {RUN_ID}")
    print(f"{'='*65}\n")

    conn    = get_conn()
    tickers = leer_tickers(conn)
    hoy     = datetime.today().date()

    print(f"  Tickers en catálogo: {len(tickers)}\n")

    # ── Paso 1: descargar precios nuevos para todos los tickers
    print("  [1/3] Descargando precios desde Polygon...")
    for t in tickers:
        ticker = t["ticker"]

        ultima = ultima_fecha_en_db(conn, ticker)

        if ultima is None:
            # Primera vez — bajar histórico completo
            desde = (hoy - timedelta(days=DIAS_HIST)).strftime("%Y-%m-%d")
            modo  = "histórico completo"
        else:
            # Ya tenemos datos — solo pedir desde el día siguiente
            desde = (ultima + timedelta(days=1)).strftime("%Y-%m-%d")
            modo  = f"incremental desde {desde}"

        hasta = hoy.strftime("%Y-%m-%d")

        if desde > hasta:
            print(f"  ✓ {ticker:6} — ya actualizado")
            continue

        print(f"  → {ticker:6} ({modo})", end=" ")
        df = descargar_precios(ticker, desde, hasta)
        n  = insertar_precios(conn, ticker, df)
        print(f"→ {n} filas insertadas")

    # ── Paso 2: cargar SPY como benchmark
    print("\n  [2/3] Cargando benchmark SPY...")
    df_spy = cargar_serie(conn, BENCHMARK)
    if df_spy.empty:
        print("  ✗ SPY sin datos — abortando cálculo de indicadores")
        conn.close()
        return
    print(f"  ✓ SPY: {len(df_spy)} días de historia")

    # ── Paso 3: calcular indicadores para cada ticker
    print("\n  [3/3] Calculando indicadores...")
    resultados = []

    for t in tickers:
        ticker = t["ticker"]
        if ticker == BENCHMARK:
            continue

        df_ticker = cargar_serie(conn, ticker)
        indicadores = calcular_indicadores(df_ticker, df_spy)

        if indicadores is None:
            print(f"  ⚠ {ticker:6} — datos insuficientes")
            continue

        fila = {**t, **indicadores, "run_id": RUN_ID}
        resultados.append(fila)
        print(f"  ✓ {ticker:6} | RS={indicadores['rs_vs_spy']:.3f} "
              f"| RSI_sem={indicadores['rsi_rs_semanal']:.1f} "
              f"| ret_3m={indicadores['ret_3m']:.1f}%")

    # ── Resumen
    print(f"\n{'─'*65}")
    print(f"  Tickers procesados: {len(resultados)}")

    if resultados:
        df_res = pd.DataFrame(resultados)
        top5 = df_res[df_res["tipo"]=="industria"].nlargest(5, "rsi_rs_semanal")[["ticker","industria","rs_vs_spy","rsi_rs_semanal","ret_3m"]]
        print(f"\n  Top 5 por RSI semanal (industrias):")
        print(top5.to_string(index=False))

        # ── Guardar en sector_snapshot
        print("\n  Guardando snapshot en DB...")
        guardar_snapshot(conn, resultados)

    print(f"\n{'='*65}")
    print(f"  Pipeline completo — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*65}\n")

    conn.close()
    return resultados


if __name__ == "__main__":
    main()


  SECTOR PRECIOS — 2026-04-05 21:14  |  run: 20260405_2114

  Tickers en catálogo: 63

  [1/3] Descargando precios desde Polygon...
  → SPY    (incremental desde 2026-04-03)   ⚠ Sin datos: SPY
→ 0 filas insertadas
  → FIVG   (incremental desde 2024-07-20)   ⚠ Sin datos: FIVG
→ 0 filas insertadas
  → NERD   (incremental desde 2026-04-03)   ⚠ Sin datos: NERD
→ 0 filas insertadas
  → SOCL   (incremental desde 2026-04-03)   ⚠ Sin datos: SOCL
→ 0 filas insertadas
  → FTXG   (incremental desde 2026-04-03)   ⚠ Sin datos: FTXG
→ 0 filas insertadas
  → JHMS   (histórico completo) 
  ⏳ Rate limit en JHMS (intento 1/3) — esperando 60s...
  ⚠ Sin datos: JHMS
→ 0 filas insertadas
  → PBJ    (incremental desde 2026-04-03)   ⚠ Sin datos: PBJ
→ 0 filas insertadas
  → AWAY   (incremental desde 2026-04-03)   ⚠ Sin datos: AWAY
→ 0 filas insertadas
  → CARZ   (incremental desde 2026-04-03)   ⚠ Sin datos: CARZ
→ 0 filas insertadas
  → JETS   (incremental desde 2026-04-03)   ⚠ Sin datos: JETS
→ 0 filas ins

  ✓ XLE    | RS=0.090 | RSI_sem=80.5 | ret_3m=32.5%
  ✓ XLF    | RS=0.076 | RSI_sem=31.1 | ret_3m=-9.6%
  ✓ XLI    | RS=0.250 | RSI_sem=72.0 | ret_3m=5.6%
  ✓ XLB    | RS=0.077 | RSI_sem=68.6 | ret_3m=11.2%
  ✓ XLRE   | RS=0.063 | RSI_sem=69.0 | ret_3m=3.1%
  ✓ XLV    | RS=0.224 | RSI_sem=47.4 | ret_3m=-5.2%
  ✓ XLK    | RS=0.207 | RSI_sem=39.4 | ret_3m=-5.5%
  ✓ XLU    | RS=0.071 | RSI_sem=68.6 | ret_3m=8.6%

─────────────────────────────────────────────────────────────────
  Tickers procesados: 60

  Top 5 por RSI semanal (industrias):
ticker                industria  rs_vs_spy  rsi_rs_semanal    ret_3m
   FAN                   Eólica   0.037906       91.604094 21.446019
   XOP Exploración y producción   0.270985       79.606538 40.757168
   OIH     Servicios petroleros   0.608466       79.218810 40.130632
  AMLP    Midstream / pipelines   0.079853       77.422167 11.378137
  SRVR    Data centers / torres   0.048717       75.760633 11.557263

  Guardando snapshot en DB...
  ✓ 60 fila